# 1. Load data by databricks autoloader

### 1.1. Autoloader function for all the csv files

In [0]:
from pyspark.sql import functions as F

storage_account = "azuresales"

def autoload_to_bronze(entity_name, raw_subfolder, target_table):
    raw_path        = f"abfss://raw@{storage_account}.dfs.core.windows.net/{raw_subfolder}/"
    checkpoint_path = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_checkpoints/{entity_name}/"
    schema_location = f"abfss://bronze@{storage_account}.dfs.core.windows.net/_schemas/{entity_name}/"

    df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("cloudFiles.useManagedFileEvents", "true")
          .option("cloudFiles.schemaLocation", schema_location)
          .option("header", "true")
          .load(raw_path)
          .withColumn("_source_file", F.col("_metadata.file_path"))
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_entity", F.lit(entity_name))
         )

    query = (df.writeStream
             .format("delta")
             .option("checkpointLocation", checkpoint_path)
             .trigger(availableNow=True)
             .toTable(target_table)
            )
    return query

### 1.2. Listing the entities

In [0]:
entities = [
    {"entity_name": "customers",    "raw_subfolder": "dimensions/customers",  "target_table": "azuresalesdatabricks.bronze.customers"},
    {"entity_name": "products",     "raw_subfolder": "dimensions/products",   "target_table": "azuresalesdatabricks.bronze.products"},
    {"entity_name": "stores",       "raw_subfolder": "dimensions/stores",     "target_table": "azuresalesdatabricks.bronze.stores"},
    {"entity_name": "sales_reps",   "raw_subfolder": "dimensions/sales_reps", "target_table": "azuresalesdatabricks.bronze.sales_reps"},
    {"entity_name": "sales_orders", "raw_subfolder": "facts/sales_orders",    "target_table": "azuresalesdatabricks.bronze.sales_orders"},
    {"entity_name": "order_items",  "raw_subfolder": "facts/order_items",     "target_table": "azuresalesdatabricks.bronze.order_items"},
    {"entity_name": "returns",      "raw_subfolder": "facts/returns",         "target_table": "azuresalesdatabricks.bronze.returns"},
]

### 1.3. Data ingestion by autoloader function

In [0]:
for e in entities:
    print(f"Ingesting {e['entity_name']}...")
    query = autoload_to_bronze(e["entity_name"], e["raw_subfolder"], e["target_table"])
    query.awaitTermination()   # block until this entity's batch finishes before starting the next
    print(f"Done: {e['entity_name']}")